In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('train.csv')

In [ ]:
df.head()

In [ ]:
new_df= df.sample(50000)

In [ ]:
new_df.isnull().sum()

In [ ]:
new_df.duplicated().sum()

In [ ]:
new_df['q1_length']= new_df['question1'].str.len()
new_df['q2_length']= new_df['question2'].str.len()

In [ ]:
new_df.head()

In [ ]:
new_df["q1_num_word"]= new_df['question1'].apply(lambda row: len(row.split(" ")))
new_df["q2_num_word"]= new_df['question2'].apply(lambda row: len(row.split(" ")))

In [ ]:
new_df.head()

In [ ]:
new_df['common_words'] = new_df.apply(
    lambda row: len(set(row['question1'].lower().split())
                    & set(row['question2'].lower().split())),
    axis=1
)

In [ ]:
new_df.head()

In [ ]:
def total_words(row):
    w1 = set(map(lambda word: word.lower().strip(),
             row['question1'].split(" ")))
    w2 = set(map(lambda word: word.lower().strip(),
             row['question2'].split(" ")))
    return (len(w1) + len(w2))

In [ ]:
new_df['word_total'] = new_df.apply(total_words, axis=1)
new_df.head()

In [ ]:
new_df['word_share'] = round(new_df['common_words']/new_df['word_total'], 2)
new_df.head()

In [ ]:
%store new_df

In [ ]:
# Analysis of features
sns.displot(new_df['q1_length'])
print('minimum characters', new_df['q1_length'].min())
print('maximum characters', new_df['q1_length'].max())
print('average num of characters', int(new_df['q1_length'].mean()))

In [ ]:
sns.displot(new_df['q2_length'])
print('minimum characters', new_df['q2_length'].min())
print('maximum characters', new_df['q2_length'].max())
print('average num of characters', int(new_df['q2_length'].mean()))

In [ ]:
sns.displot(new_df['q1_num_word'])
print('minimum words', new_df['q1_num_word'].min())
print('maximum words', new_df['q1_num_word'].max())
print('average num of words', int(new_df['q1_num_word'].mean()))

In [ ]:
sns.displot(new_df['q2_num_word'])
print('minimum words', new_df['q2_num_word'].min())
print('maximum words', new_df['q2_num_word'].max())
print('average num of words', int(new_df['q2_num_word'].mean()))

In [ ]:
# common words
sns.distplot(new_df[new_df['is_duplicate'] == 0]
             ['common_words'], label='non duplicate')
sns.distplot(new_df[new_df['is_duplicate'] == 1]
             ['common_words'], label='duplicate')
plt.legend()
plt.show()

In [ ]:
# word share
sns.distplot(new_df[new_df['is_duplicate'] == 0]
             ['word_share'], label='non duplicate')
sns.distplot(new_df[new_df['is_duplicate'] == 1]
             ['word_share'], label='duplicate')
plt.legend()
plt.show()

In [ ]:
ques_df= new_df[['question1', 'question2']]

In [ ]:
final_df = new_df[['id',	'qid1',	'qid2',	'q1_length', 'q2_length','q1_num_word',	'q2_num_word',	'common_words',	'word_total'	,'word_share' , 'is_duplicate']]

In [ ]:
final_df.head()

In [ ]:
ques_df.head()

In [ ]:
question= list(ques_df['question1']) + list(ques_df['question2'])

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv= CountVectorizer(max_features= 5000)
q1_arr, q2_arr = np.vsplit(cv.fit_transform(question).toarray(), 2)

In [ ]:
q1_arr.shape

In [ ]:
temp_df1 = pd.DataFrame(q1_arr, index=ques_df.index)
temp_df2 = pd.DataFrame(q2_arr, index=ques_df.index)
temp_df = pd.concat([temp_df1, temp_df2], axis=1)

In [ ]:
temp_df.head()

In [ ]:
new_final_df= pd.concat([temp_df, final_df], axis= 1)

In [ ]:
new_final_df.head()

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test= train_test_split(new_final_df.iloc[:, :-1], new_final_df.iloc[:, -1], test_size= 0.2, random_state= 2)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ensure all column names are strings to avoid mixed-type feature name error
x_train.columns = x_train.columns.astype(str)
x_test.columns = x_test.columns.astype(str)

rf = RandomForestClassifier()
rf = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)
rf.fit(x_train.astype(np.float32), y_train)
y_pred= rf.predict(x_test)
accuracy_score(y_test, y_pred)